# AMLAnnotator Tutorial — Pan-Leukemia Model (AML + B-ALL)

This tutorial demonstrates how to use the **"pan_leukemia" model** to classify cells as **malignant vs normal**, on either AML or B-ALL scRNA-seq data.

Unlike the default `"full"` model (AML-only, three levels — see `tutorial.ipynb`), `"pan_leukemia"` currently supports **Level 1 only**: malignant vs normal. Blast group and LSC type for this model are planned for a future release.

## Installation

```bash
pip install git+https://github.com/sinakane/AMLAnnotator.git
```

## 1. Load your data

Works the same way regardless of model: any AnnData object with standard gene symbols in `var_names` and raw counts (in `.X` or a named layer). No disease label or other metadata is required — the model infers everything from expression alone.

This example uses an AML sample, but a B-ALL sample works identically.

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Load your AML or B-ALL scRNA-seq data
adata = sc.read_h5ad("your_data.h5ad")
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")

## 2. Initialize AMLAnnotator with `model="pan_leukemia"`

Call `AMLAnnotator.list_models()` any time to see what's available without downloading anything. Models are downloaded automatically, once, the first time you request a given `model=`.

In [ ]:
from amlannotator import AMLAnnotator

AMLAnnotator.list_models()

annotator = AMLAnnotator(model="pan_leukemia")

# View model information
info = annotator.get_model_info()
print(f"\nModel: {info['model']}")
for level, details in info["levels"].items():
    print(f"\n{level}")
    print(f"  Classes: {details['classes']}")
    print(f"  Features: {details['n_features']}")
    print(f"  Accuracy: {details['accuracy']:.4f}")

## 3. Annotate

Same `.annotate()` call as the full model — the method checks which levels the loaded model supports and only populates those columns. For `"pan_leukemia"`, that's malignant/normal only; calling `.annotate()` prints a note confirming blast group / LSC type were skipped, rather than erroring.

**Parameters:**
- `layer` — Layer containing raw counts (e.g. `'raw_counts'`). If `None`, uses `.X`.
- `copy` — If `True`, return a copy instead of modifying in place.
- `malignant_prob_threshold` — Cells classified as normal but with an averaged malignant probability above this threshold are reclassified as malignant. Default `0.3`.

In [ ]:
# Annotate — specify the layer containing raw counts
adata = annotator.annotate(adata, layer="raw_counts")

# If raw counts are in .X:
# adata = annotator.annotate(adata)

## 4. Explore results

In [ ]:
print(adata.obs["aml_malignant_normal"].value_counts())
print(f"\nMean confidence: {adata.obs['aml_malignant_confidence'].mean():.3f}")

# blast group / LSC type columns are NOT added by this model
print("\nblast/LSC columns present:",
      any(c.startswith("aml_blast") or c.startswith("aml_lsc") for c in adata.obs.columns))

### UMAP visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata, color="aml_malignant_normal", ax=axes[0], show=False,
           title="Malignant vs Normal")
sc.pl.umap(adata, color="aml_malignant_confidence", ax=axes[1], show=False,
           title="Confidence", cmap="viridis")
plt.tight_layout()

## 5. Save results

In [ ]:
adata.write("my_data_annotated.h5ad")

cols = [c for c in adata.obs.columns if c.startswith("aml_")]
adata.obs[cols].to_csv("malignant_normal_annotations.csv")
print(f"Saved {len(cols)} annotation columns")

## Switching to the full AML model

If you're working with AML data specifically and want blast group / LSC type as well, use `model="full"` instead — see `tutorial.ipynb`. The two models are independent; you can load and use both in the same session.